First we need the state abbreviations for all 50 states and the District of Colombia

In [1]:
states = [
    "AL","AK","AZ","AR","CA","CO","CT","DC","DE","FL","GA",
    "HI","ID","IL","IN","IA","KS","KY","LA","ME","MD",
    "MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ",
    "NM","NY","NC","ND","OH","OK","OR","PA","RI","SC",
    "SD","TN","TX","UT","VT","VA","WA","WV","WI","WY"
]

Then we can collect all of the agency codes (ORI) for the states.

In [2]:
import requests
import pandas as pd

API_KEY = "HdJoOraec3MgSMMu6Q1Y5dIC8UkTcpiqKSNIQTDn"


def get_state_agencies(state):

    url = f"https://api.usa.gov/crime/fbi/cde/agency/byStateAbbr/{state}"
    params = {"api_key": API_KEY}

    r = requests.get(url, params=params)

    if r.status_code != 200:
        print(f"Failed for {state}")
        return None

    return r.json()

In [3]:
all_agencies = []

for state in states:

    data = get_state_agencies(state)

    if data is not None:
        for county, agencies in data.items():
            for agency in agencies:
                agency['county_name'] = county
                all_agencies.append(agency)

df_agencies = pd.DataFrame(all_agencies)

In [4]:
df_agencies = df_agencies[df_agencies['is_nibrs'] == True]

Don't forget to save your progress at each step! These FBI calls sometimes take a long time. Make sure to back up your work at each step.

In [5]:
df_agencies.to_csv("all_US_agencies.csv", index=False)

In [6]:
ori_list = df_agencies['ori'].dropna().unique().tolist()

print("Total agencies:", len(ori_list))

Total agencies: 16155


Use this function to collect all of the arrest counts for the 16,155 agencies.

In [7]:
def get_agency_arrests_summary(ori):

    url = f"https://api.usa.gov/crime/fbi/cde/arrest/agency/{ori}/all"

    params = {
        "type": "counts",
        "from": "01-2024",
        "to": "12-2024",
        "api_key": API_KEY
    }

    try:
        r = requests.get(url, params=params, timeout=10)

        if r.status_code != 200:
            return None

        data = r.json()

        if not data.get('actuals'):
            return None

        agency_label = next(iter(data['actuals']))

        actuals = data['actuals'].get(agency_label, {})
        population_dict = data['populations']['population']

        agency_name = agency_label.replace(" Arrests", "")
        population = population_dict.get(agency_name, {})

        total_arrests = sum(
            v if isinstance(v, (int, float)) else 0
            for v in actuals.values()
        )

        pop_values = [v for v in population.values() if isinstance(v, (int, float))]
        population_value = max(pop_values) if pop_values else None

        return {
            'ori': ori,
            'Arrests_2024': total_arrests,
            'Population_2024': population_value
        }

    except:
        return None

I added a status bar and a error check to this function. I ran into many choke points when trying to collect so much data at the same time.

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import pandas as pd

def collect_all_summaries(ori_list, max_workers=10):

    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:

        futures = {
            executor.submit(get_agency_arrests_summary, ori): ori
            for ori in ori_list
        }

        for future in tqdm(as_completed(futures), total=len(futures)):

            try:
                result = future.result()

                if result is not None:
                    results.append(result)

            except Exception as e:
                print(f"Error: {e}")

    return pd.DataFrame(results)

WARNING! This next line of code will take about 15 minutes to complete its task.

In [9]:
summed_US_df = collect_all_summaries(ori_list, max_workers=10)

100%|██████████| 16155/16155 [18:56<00:00, 14.21it/s] 


In [10]:
final_df = pd.merge(df_agencies, summed_US_df, on='ori')

In [11]:
final_df.head()

,ori,counties,is_nibrs,latitude,longitude,state_abbr,state_name,agency_name,agency_type_name,nibrs_start_date,county_name,Arrests_2024,Population_2024
0,AL0430000,LEE,True,32.604064,-85.353048,AL,Alabama,Lee County Sheriff's Office,County,2012-04-01,LEE,3121,61961.0
1,AL0430200,LEE,True,32.604064,-85.353048,AL,Alabama,Opelika Police Department,City,2021-01-01,LEE,1454,34448.0
2,AL0430100,LEE,True,32.608086,-85.475136,AL,Alabama,Auburn Police Department,City,2020-12-01,LEE,2827,83756.0
3,AL0070300,BIBB,True,33.015893,-87.127148,AL,Alabama,West Blocton Police Department,City,2020-01-01,BIBB,49,1181.0
4,AL0070200,BIBB,True,33.015893,-87.127148,AL,Alabama,Brent Police Department,City,2020-01-01,BIBB,155,2586.0


Then I realized I don't just need arrests, I need the actual total crime values, too. Now we will collect the total number of violent crimes and also property crimes for each agency.

In [12]:
def get_agency_offenses_summary(ori, crime_type):
    # crime_type = 'V' (violent) or 'P' (property)

    url = f"https://api.usa.gov/crime/fbi/cde/summarized/agency/{ori}/{crime_type}"

    params = {
        "from": "01-2024",
        "to": "12-2024",
        "api_key": API_KEY
    }

    try:
        r = requests.get(url, params=params, timeout=10)

        if r.status_code != 200:
            return None

        data = r.json()

        # --- KEY DIFFERENCE ---
        if not data.get('offenses') or not data['offenses'].get('actuals'):
            return None

        actuals_dict = data['offenses']['actuals']

        # find the "Offenses" key (ignore "Clearances")
        offense_keys = [k for k in actuals_dict.keys() if "Offenses" in k]

        if not offense_keys:
            return None

        agency_label = offense_keys[0]

        monthly_data = actuals_dict.get(agency_label, {})

        total_offenses = sum(
            v if isinstance(v, (int, float)) else 0
            for v in monthly_data.values()
        )

        return {
            'ori': ori,
            f'{crime_type}_crimes_2024': total_offenses
        }

    except:
        return None

In [13]:
def collect_offense_data(ori_list, crime_type, max_workers=10):

    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:

        futures = {
            executor.submit(get_agency_offenses_summary, ori, crime_type): ori
            for ori in ori_list
        }

        for future in tqdm(as_completed(futures), total=len(futures)):

            try:
                result = future.result()

                if result is not None:
                    results.append(result)

            except Exception as e:
                print(f"Error: {e}")

    return pd.DataFrame(results)

WARNING! These two lines of code will take about 30 minutes to run.

In [14]:
violent_df = collect_offense_data(ori_list, 'V', max_workers=10)
property_df = collect_offense_data(ori_list, 'P', max_workers=10)

100%|██████████| 16155/16155 [35:00<00:00,  7.69it/s]  


In [15]:
crime_df = final_df.merge(violent_df, on='ori', how='left')
crime_df = crime_df.merge(property_df, on='ori', how='left')

Document and save your work as you go!

In [16]:
crime_df.to_csv('US_crime_data_2024.csv', index=False)

In [17]:
county_crime_df = (
    crime_df
    .groupby(['state_abbr', 'state_name', 'county_name'], as_index=False)
    .agg(
        latitude=('latitude', 'mean'),
        longitude=('longitude', 'mean'),
        Population_2024=('Population_2024', 'sum'),
        Arrests_2024=('Arrests_2024', 'sum'),
        Violent_Crimes_2024=('V_crimes_2024', 'sum'),
        Property_Crimes_2024=('P_crimes_2024', 'sum')
    )
)

county_crime_df.head()

,state_abbr,state_name,county_name,latitude,longitude,Population_2024,Arrests_2024,Violent_Crimes_2024,Property_Crimes_2024
0,AK,Alaska,ALEUTIANS WEST,51.959447,178.338813,4204.0,42,6,4.0
1,AK,Alaska,BETHEL,60.928916,-160.153350,6280.0,748,104,145.0
2,AK,Alaska,BRISTOL BAY,58.732389,-156.986661,856.0,28,7,6.0
3,AK,Alaska,DILLINGHAM,59.824816,-158.602233,2088.0,35,7,1.0
4,AK,Alaska,FAIRBANKS NORTH STAR,64.797240,-147.535115,34166.0,1052,237,1157.0


Add the arrest and crime rates per 100k

In [18]:
county_crime_df['Arrest_Rate'] = (
    county_crime_df['Arrests_2024'] /
    county_crime_df['Population_2024']
) * 100000


county_crime_df['Violent_Crime_Rate'] = (
    county_crime_df['Violent_Crimes_2024'] /
    county_crime_df['Population_2024']
) * 100000

county_crime_df['Property_Crime_Rate'] = (
    county_crime_df['Property_Crimes_2024'] /
    county_crime_df['Population_2024']
) * 100000

Fix the one longitude I can see is obviously wrong... (missing negative)

In [19]:
county_crime_df.loc[county_crime_df['county_name'] == 'ALEUTIANS WEST', 'longitude'] = -178.338813

Make some rough adjustments so that we can plot the data.

In [20]:
county_crime_df = county_crime_df.dropna(subset=['latitude', 'longitude'])
county_crime_df = county_crime_df.fillna(0)

Attempt a plot!

In [22]:
import plotly.express as px

fig = px.scatter_mapbox(
    county_crime_df,
    lat="latitude",
    lon="longitude",
    size="Violent_Crime_Rate",
    color="Violent_Crime_Rate",
    hover_name="county_name",
    hover_data=["county_name", "state_abbr", "Violent_Crime_Rate"],
    color_continuous_scale="Viridis",
    size_max=40,
    zoom=3
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r":0,"t":0,"l":0,"b":0}
)

fig.show()

Let's look at Property Crime, too.

In [23]:
import plotly.express as px

fig = px.scatter_mapbox(
    county_crime_df,
    lat="latitude",
    lon="longitude",
    size="Property_Crime_Rate",
    color="Property_Crime_Rate",
    hover_name="county_name",
    hover_data=["county_name", "state_abbr", "Property_Crime_Rate"],
    color_continuous_scale="Viridis",
    size_max=40,
    zoom=3
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r":0,"t":0,"l":0,"b":0}
)

fig.show()

Make that final save point.

In [24]:
county_crime_df.to_csv('county_crime_data_2024.csv', index=False)